# Heap Volume Calculation
**Author: Aayush Yadav**
Exploration of the provided test data and an independent check of the volume computed by the QGIS Processing script.

**Inputs Required :** `heap-dem.tif`, `base_dem.tif`, `heap/polygons.shp` (make sure the these files present in the same folder as the notebook)

In [37]:
# Importing dependencies

import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon, box

### Inspecting DEMs
Read each DEM once

In [24]:
def read_dem(file):
    with rasterio.open(file) as src:
        return {
            "array": src.read(1, masked=True),
            "transform": src.transform,
            "crs": src.crs,
            "res": src.res,
            "shape": (src.height, src.width),
            "nodata": src.nodata,
            "bounds": src.bounds,
        }

In [25]:
old_dem = read_dem("base_dem.tif")
new_dem = read_dem("heap-dem.tif")

In [26]:
for dem in (old_dem, new_dem):
    a = dem["array"]
    print(f"CRS {dem['crs'].to_epsg()} | {dem['shape'][1]} x {dem['shape'][0]} px | "
          f"pixel {dem['res'][0]:.4f} m | NoData {dem['nodata']} | "
          f"elev {a.min():.2f}–{a.max():.2f} m | NoData px {np.ma.count_masked(a)}")

CRS 32644 | 5757 x 3196 px | pixel 0.0810 m | NoData -32767.0 | elev 13.89–18.00 m | NoData px 0
CRS 32644 | 5757 x 3196 px | pixel 0.0810 m | NoData -32767.0 | elev 13.89–30.70 m | NoData px 0


### Loading the heap

In [27]:
heap = gpd.read_file("heap/polygons.shp")
print(heap.crs)

EPSG:4326


### Reprojecting heap
The polygons are in EPSG:4326 (degrees). They are reprojected to the rasters' CRS

In [28]:
heap = heap.to_crs(new_dem["crs"])

In [29]:
# Fucntion to check whether the heap lies within the DEM

def check_limit(vec_heap, raster_dem):
    return vec_heap.within(box(*raster_dem["bounds"]))

In [30]:
chk = check_limit(heap, old_dem)
if not chk.all():
    print("Heaps outside the DEM:", list(heap.index[~chk]))

### Rasterizing the heap and checking mask

In [31]:
from rasterio.features import geometry_mask

geom = heap.geometry.iloc[0]
inside = geometry_mask([geom], out_shape=new_dem["array"].shape, transform=new_dem["transform"], invert=True)

pixel_area = abs(new_dem["res"][0] * new_dem["res"][1])
print("pixels inside :", inside.sum())
print("expected      :", heap.area.iloc[0] / pixel_area)

pixels inside : 584045
expected      : 584014.9566101244


### Calculating Volume

In [32]:
diff = (new_dem["array"] - old_dem["array"])[inside].compressed().astype(np.float64)

cut = abs(diff[diff<0]*pixel_area)
fill = diff[diff>0]*pixel_area
volume = (fill.sum() - cut.sum())

print(f'Cut Volume (m_3): {cut.sum()}')
print(f'Fill Volume (m_3): {fill.sum()}')
print(f'Net Volume (m_3): {volume}')

Cut Volume (m_3): 0.0
Fill Volume (m_3): 16212.558252788875
Net Volume (m_3): 16212.558252788875


### For all Heaps

In [40]:
rows = []
for i, geom in enumerate(heap.geometry):
    inside = geometry_mask([geom], out_shape=new_dem["array"].shape, transform=new_dem["transform"], invert=True)
    d = (new_dem["array"] - old_dem["array"])[inside].compressed().astype(np.float64)

    cut = abs(d[d<0]*pixel_area)
    fill = d[d>0]*pixel_area
    volume = (fill.sum() - cut.sum())
    
    rows.append({"heap": i, "fill_m3": fill.sum(), "cut_m3": cut.sum(), "net_m3": volume})

results = pd.DataFrame(rows).set_index("heap")
results.loc["total"] = results.sum()
results.round(2)

,fill_m3,cut_m3,net_m3
heap,,,
0,16212.56,0.0,16212.56
1,5928.11,0.0,5928.11
2,6502.59,0.0,6502.59
total,28643.26,0.0,28643.26
